# Random Forest Classifier 

## Dataset Overview
The mushroom dataset is used to classify mushrooms as either edible (e) or poisonous (p) based on various physical characteristics.

In [2]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Data Loading and Preprocessing

In [3]:
# Load the dataset
df = pd.read_csv('../../../Datasets/mushrooms.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst Few Rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nClass Distribution:")
print(df['class'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '../../../Datasets/mushrooms.csv'

In [ ]:
# Encode categorical variables
# Create a copy of the dataframe for processing
df_encoded = df.copy()

# Separate features and target
X = df_encoded.drop('class', axis=1)
y = df_encoded['class']

# Encode all columns using LabelEncoder
le_dict = {}
for column in X.columns:
    le = LabelEncoder()
    X[column] = le.fit_transform(X[column])
    le_dict[column] = le

# Encode target variable
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print("Encoded Features Shape:", X.shape)
print("Target Classes:", le_target.classes_)
print("\nFeature Names:", X.columns.tolist())

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])
print("\nTraining set class distribution:")
print(pd.Series(y_train).value_counts())
print("\nTesting set class distribution:")
print(pd.Series(y_test).value_counts())

## Model 1: Bagging (Bootstrap Aggregating)
Bagging reduces variance by training multiple models on random subsets of the training data and averaging their predictions.

In [ ]:
# Train Bagging Classifier (equivalent to Random Forest with bagging)
bagging_model = BaggingClassifier(
    estimator=RandomForestClassifier(random_state=42),
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

bagging_model.fit(X_train, y_train)

# Make predictions
y_pred_bagging = bagging_model.predict(X_test)
y_pred_proba_bagging = bagging_model.predict_proba(X_test)[:, 1]

print("Bagging Model trained successfully!")

### Performance Metrics - Bagging

In [ ]:
# Calculate performance metrics for Bagging
bagging_accuracy = accuracy_score(y_test, y_pred_bagging)
bagging_precision = precision_score(y_test, y_pred_bagging)
bagging_recall = recall_score(y_test, y_pred_bagging)
bagging_f1 = f1_score(y_test, y_pred_bagging)
bagging_roc_auc = roc_auc_score(y_test, y_pred_proba_bagging)

print("=" * 50)
print("BAGGING MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"Accuracy:  {bagging_accuracy:.4f}")
print(f"Precision: {bagging_precision:.4f}")
print(f"Recall:    {bagging_recall:.4f}")
print(f"F1-Score:  {bagging_f1:.4f}")
print(f"ROC-AUC:   {bagging_roc_auc:.4f}")
print("\n" + "=" * 50)
print("Classification Report - Bagging")
print("=" * 50)
print(classification_report(y_test, y_pred_bagging, target_names=le_target.classes_))

In [ ]:
# Confusion Matrix - Bagging
cm_bagging = confusion_matrix(y_test, y_pred_bagging)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_bagging, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le_target.classes_, yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix - Bagging Classifier', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

print(f"True Negatives: {cm_bagging[0, 0]}")
print(f"False Positives: {cm_bagging[0, 1]}")
print(f"False Negatives: {cm_bagging[1, 0]}")
print(f"True Positives: {cm_bagging[1, 1]}")

In [ ]:
# ROC Curve - Bagging
fpr_bagging, tpr_bagging, _ = roc_curve(y_test, y_pred_proba_bagging)

plt.figure(figsize=(8, 6))
plt.plot(fpr_bagging, tpr_bagging, label=f'Bagging (AUC = {bagging_roc_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Bagging Classifier', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Model 2: Boosting - AdaBoost
AdaBoost reduces both bias and variance by sequentially training models, with each model focusing on the misclassified instances from the previous model.

In [ ]:
# Train AdaBoost Classifier
adaboost_model = AdaBoostClassifier(
    estimator=RandomForestClassifier(n_estimators=10, random_state=42),
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

adaboost_model.fit(X_train, y_train)

# Make predictions
y_pred_adaboost = adaboost_model.predict(X_test)
y_pred_proba_adaboost = adaboost_model.predict_proba(X_test)[:, 1]

print("AdaBoost Model trained successfully!")

### Performance Metrics - AdaBoost

In [ ]:
# Calculate performance metrics for AdaBoost
adaboost_accuracy = accuracy_score(y_test, y_pred_adaboost)
adaboost_precision = precision_score(y_test, y_pred_adaboost)
adaboost_recall = recall_score(y_test, y_pred_adaboost)
adaboost_f1 = f1_score(y_test, y_pred_adaboost)
adaboost_roc_auc = roc_auc_score(y_test, y_pred_proba_adaboost)

print("=" * 50)
print("ADABOOST MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"Accuracy:  {adaboost_accuracy:.4f}")
print(f"Precision: {adaboost_precision:.4f}")
print(f"Recall:    {adaboost_recall:.4f}")
print(f"F1-Score:  {adaboost_f1:.4f}")
print(f"ROC-AUC:   {adaboost_roc_auc:.4f}")
print("\n" + "=" * 50)
print("Classification Report - AdaBoost")
print("=" * 50)
print(classification_report(y_test, y_pred_adaboost, target_names=le_target.classes_))

In [ ]:
# Confusion Matrix - AdaBoost
cm_adaboost = confusion_matrix(y_test, y_pred_adaboost)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_adaboost, annot=True, fmt='d', cmap='Greens', 
            xticklabels=le_target.classes_, yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix - AdaBoost Classifier', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

print(f"True Negatives: {cm_adaboost[0, 0]}")
print(f"False Positives: {cm_adaboost[0, 1]}")
print(f"False Negatives: {cm_adaboost[1, 0]}")
print(f"True Positives: {cm_adaboost[1, 1]}")

## Model 3: Boosting - Gradient Boosting
Gradient Boosting builds an ensemble of decision trees sequentially, where each new tree is added to correct the residuals of the previous predictions.

In [ ]:
# Train Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

gb_model.fit(X_train, y_train)

# Make predictions
y_pred_gb = gb_model.predict(X_test)
y_pred_proba_gb = gb_model.predict_proba(X_test)[:, 1]

print("Gradient Boosting Model trained successfully!")

### Performance Metrics - Gradient Boosting

In [ ]:
# Calculate performance metrics for Gradient Boosting
gb_accuracy = accuracy_score(y_test, y_pred_gb)
gb_precision = precision_score(y_test, y_pred_gb)
gb_recall = recall_score(y_test, y_pred_gb)
gb_f1 = f1_score(y_test, y_pred_gb)
gb_roc_auc = roc_auc_score(y_test, y_pred_proba_gb)

print("=" * 50)
print("GRADIENT BOOSTING MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"Accuracy:  {gb_accuracy:.4f}")
print(f"Precision: {gb_precision:.4f}")
print(f"Recall:    {gb_recall:.4f}")
print(f"F1-Score:  {gb_f1:.4f}")
print(f"ROC-AUC:   {gb_roc_auc:.4f}")
print("\n" + "=" * 50)
print("Classification Report - Gradient Boosting")
print("=" * 50)
print(classification_report(y_test, y_pred_gb, target_names=le_target.classes_))

In [ ]:
# Confusion Matrix - Gradient Boosting
cm_gb = confusion_matrix(y_test, y_pred_gb)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=le_target.classes_, yticklabels=le_target.classes_, ax=ax)
ax.set_title('Confusion Matrix - Gradient Boosting Classifier', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

print(f"True Negatives: {cm_gb[0, 0]}")
print(f"False Positives: {cm_gb[0, 1]}")
print(f"False Negatives: {cm_gb[1, 0]}")
print(f"True Positives: {cm_gb[1, 1]}")

## Model Comparison

In [ ]:
# Create Comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': ['Bagging', 'AdaBoost', 'Gradient Boosting'],
    'Accuracy': [bagging_accuracy, adaboost_accuracy, gb_accuracy],
    'Precision': [bagging_precision, adaboost_precision, gb_precision],
    'Recall': [bagging_recall, adaboost_recall, gb_recall],
    'F1-Score': [bagging_f1, adaboost_f1, gb_f1],
    'ROC-AUC': [bagging_roc_auc, adaboost_roc_auc, gb_roc_auc]
})

print("=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

In [ ]:
# Visualize Model Comparison - Bar Plot
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
models = comparison_df['Model'].tolist()
colors = ['#3498db', '#2ecc71', '#e74c3c']

# Plot each metric
for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    values = comparison_df[metric].tolist()
    bars = ax.bar(models, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel(metric, fontsize=11, fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Remove the extra subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

In [ ]:
# Combined ROC Curve Comparison
fpr_gb, tpr_gb, _ = roc_curve(y_test, y_pred_proba_gb)
fpr_adaboost, tpr_adaboost, _ = roc_curve(y_test, y_pred_proba_adaboost)

plt.figure(figsize=(10, 7))
plt.plot(fpr_bagging, tpr_bagging, label=f'Bagging (AUC = {bagging_roc_auc:.4f})', linewidth=2.5)
plt.plot(fpr_adaboost, tpr_adaboost, label=f'AdaBoost (AUC = {adaboost_roc_auc:.4f})', linewidth=2.5)
plt.plot(fpr_gb, tpr_gb, label=f'Gradient Boosting (AUC = {gb_roc_auc:.4f})', linewidth=2.5)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1.5)
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('ROC Curve Comparison - All Models', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Feature Importance Analysis

In [ ]:
# Extract feature importances from Gradient Boosting (best model)
feature_importance = gb_model.feature_importances_
feature_names = X.columns.tolist()

# Create a DataFrame for feature importance
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False).head(15)

# Plot Feature Importance
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'], color='#9b59b6', alpha=0.8, edgecolor='black')
ax.set_xlabel('Feature Importance Score', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Feature Importance - Gradient Boosting', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar in bars:
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2., f'{width:.4f}',
            ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nTop 15 Important Features:")
print(importance_df.to_string(index=False))